# MCD — run a Sol model on a Colab A100

Does the same thing as `models/sol/submit.sbatch`, on Colab instead of Sol:
preflight check, start vLLM, play every conversation, report.

**Needs an A100.** Runtime → Change runtime type → A100. The preflight refuses anything
smaller, because a 65,536-token window plus 16 conversations at once does not fit on a T4 or L4.

**Safe to re-run.** Everything is written to Drive after every single reply. If Colab
disconnects, open this notebook again and run every cell: finished conversations are skipped and
it picks up where it stopped. Nothing is lost but the turn that was in flight.

Run it once per model: set `MODEL` below to `qwen3_4b`, then come back and do `llama32_3b`.

## 1. Check the GPU before anything else

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader,nounits

## 2. Install vLLM

Takes a few minutes. It brings its own torch build.

In [ ]:
!pip -q install vllm

## 3. Mount Drive

The run writes here, so a disconnect does not lose it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

In [ ]:
REPO   = 'https://github.com/harpreet16/mcd-multilingual-rule-drift.git'
BRANCH = 'main'        # the run only needs main; grading happens wherever you like

import os
if not os.path.exists('/content/mcd'):
    !git clone -q --branch $BRANCH $REPO /content/mcd
%cd /content/mcd
!git log --oneline -1

## 5. Hugging Face token

`Llama-3.2-3B` is gated: accept Meta's licence on the Hugging Face website with the same
account first, or the download fails with a 403. `Qwen3-4B` is not gated.

Type the token into the box. Never paste it into a cell — a saved notebook with a token in it
is a leaked credential, and this repository is public.

In [ ]:
import getpass, os
os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')
os.environ['HF_HOME']  = '/content/hf_cache'      # model weights, not on Drive: they are large and re-downloadable

## 6. Pick the model

`qwen3_4b` first — it is not gated, so you reach a working run without fighting permissions.

In [ ]:
MODEL    = 'qwen3_4b'            # or 'llama32_3b'
VARIANTS = 'dataset/data/variants.jsonl'
OUT      = f'/content/drive/MyDrive/mcd_runs/{MODEL}'
PORT     = 8000

os.makedirs(OUT + '/logs', exist_ok=True)
print('writing to', OUT)

## 7. Preflight

The same check the Sol job runs, before any GPU time is spent. It confirms the GPU can do the job,
checks every variant against `conversations.jsonl` so what runs really is the remixer's output, and
writes one line to `jobs.jsonl` recording the dataset SHA-256, the git commit, the GPU and the
settings. That record is how anyone later proves this run matches the frozen dataset — keep it.

If it refuses, the message is written for Sol and mentions Slurm flags such as `--gres=gpu:1`.
On Colab that means the same thing as: Runtime → Change runtime type → A100, then run from cell 1 again.

In [ ]:
import subprocess, sys

pre = subprocess.run([sys.executable, '-m', 'models.sol.preflight',
                      '--model', MODEL, '--variants', VARIANTS, '--out', OUT],
                     capture_output=True, text=True)
print(pre.stdout or pre.stderr)
if pre.returncode != 0:
    raise SystemExit('preflight failed — fix the problem above before spending GPU time')

MODEL_ID, CONTEXT, DTYPE = pre.stdout.strip().splitlines()[-1].split()
print('serving', MODEL_ID, '| window', CONTEXT, '| dtype', DTYPE)

## 8. Start vLLM and wait for it

In [ ]:
import time, urllib.request

log = open(f'{OUT}/logs/vllm.log', 'w')
server = subprocess.Popen(['vllm', 'serve', MODEL_ID, '--dtype', DTYPE,
                           '--max-model-len', CONTEXT, '--gpu-memory-utilization', '0.90',
                           '--seed', '0', '--port', str(PORT)], stdout=log, stderr=log)

# First start downloads the weights, so allow 15 minutes.
ready = False
for _ in range(180):
    if server.poll() is not None:
        !tail -n 30 $OUT/logs/vllm.log
        raise SystemExit('vLLM stopped — see the log above')
    try:
        body = urllib.request.urlopen(f'http://localhost:{PORT}/v1/models', timeout=5).read().decode()
        if MODEL_ID in body:
            ready = True; break
    except Exception:
        pass
    time.sleep(5)
print('server ready' if ready else 'NOT ready after 15 minutes — check the log')

## 9. Play every conversation

Two passes, exactly like the Sol job: the second redoes anything that failed in the first.
This is the long cell — roughly 3,500 generations per model. Leave the tab open.

If it dies partway, just run this cell again.

In [ ]:
for attempt in (1, 2):
    print(f'--- pass {attempt} ---', flush=True)
    !python3 -m runner --model $MODEL --variants $VARIANTS --out $OUT --base-url http://localhost:$PORT/v1

## 10. Did it finish?

`to_do: 0` means every variant is done. Anything else: run the cell above again.

In [ ]:
!python3 -m runner --model $MODEL --variants $VARIANTS --out $OUT --base-url http://localhost:$PORT/v1 --status

## 11. Look at what the model actually said

Worth two minutes before trusting hours of GPU time. Checks for empty replies, replies cut off by
the token limit, and `<think>` blocks leaking into the answers.

In [ ]:
import json

rows = [json.loads(l) for l in open(f'{OUT}/responses.jsonl')]
replies = [r for row in rows for r in row['replies']]
cut     = sum(sum(row['cut_off']) for row in rows)
empty   = sum(1 for r in replies if not r.strip())
think   = sum(1 for r in replies if '<think>' in r)

print(f'conversations finished : {len(rows)}')
print(f'replies                : {len(replies)}')
print(f'empty replies          : {empty}      (want 0)')
print(f'cut off by token limit : {cut}      (a few is normal, many is not)')
print(f'<think> blocks leaked  : {think}      (want 0)')
print()
for row in rows[:2]:
    print(f"--- {row['conv_id']} {row['condition']}")
    for i, reply in enumerate(row['replies'][:3], 1):
        print(f'  turn {i}: {reply[:140]}')

## 12. Stop the server

Only when the status above says `to_do: 0`. Stopping early is fine too — the run resumes.

In [ ]:
server.terminate()
print('stopped. responses.jsonl and jobs.jsonl are on Drive at', OUT)

---
### After the run

Grade it with the same command whatever produced the replies:

```
python3 -m grader --responses <OUT>/responses.jsonl --out <OUT>/grades.jsonl
```

Keep `jobs.jsonl` next to `responses.jsonl`. It is the only record of which dataset version and
which commit the replies came from, and without it nobody can prove the run matches the dataset.